# 04 - Model Benchmark Sweep - Averaging over entire day

In [ ]:
!pip install pymongo catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 14.4 MB/s eta 0:00:00


In [ ]:
import os
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display
from pymongo import MongoClient
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import tensorflow as tf

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

load_dotenv(str(ROOT / ".env"), override=False)

def load_colab_secrets(keys: list[str]) -> None:
    try:
        from google.colab import userdata
    except Exception:
        return
    for key in keys:
        if os.getenv(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value

load_colab_secrets(["MONGO_URI", "MONGO_DB_NAME"])

sns.set_theme(style="whitegrid")

@dataclass
class ModelConfig:
    name: str
    type: str
    params: dict


MODEL_CONFIGS = [
    ModelConfig(name="lightgbm", type="lgbm", params= {"n_estimators": 200,"learning_rate": 0.03,"num_leaves": 15,"min_child_samples": 50,"subsample": 0.6,"colsample_bytree": 0.6,"reg_alpha": 1.0,"reg_lambda": 5.0,"n_jobs": -1,"verbose": -1,}),
    ModelConfig(name="xgboost", type="xgb", params={"n_estimators": 300, "learning_rate": 0.05, "max_depth": 6}),
    ModelConfig(name="catboost", type="cat", params={"iterations": 400, "learning_rate": 0.05, "depth": 6}),
    ModelConfig(name="random_forest", type="rf", params={"n_estimators": 400, "max_depth": 20, "n_jobs": -1}),
    ModelConfig(name="extra_trees", type="extra", params={"n_estimators": 400, "max_depth": 20, "n_jobs": -1}),
    ModelConfig(name="gradient_boosting", type="gbr", params={"n_estimators": 300, "learning_rate": 0.05}),
    ModelConfig(name="ridge_regression", type="ridge", params={"alpha": 1.0}),
    ModelConfig(name="linear_regression", type="linreg", params={}),
    ModelConfig(name="gru", type="gru", params={"units": 64, "epochs": 15, "batch_size": 64}),
    ModelConfig(name="lstm", type="lstm", params={"units": 64, "epochs": 15, "batch_size": 64}),
]

# ── Feature sets per day ─────────────────────────────────────────
# Day1: predict hours 0-24 from current state
# TOP_FEATURES_DAY1 = [
#     "european_aqi_lag_1h",
#     "european_aqi_rolling_mean_3h",
#     "european_aqi_rolling_min_3h",
#     "european_aqi_rolling_std_12h",
#     "european_aqi_rolling_std_24h",
#     "european_aqi_rolling_std_48h",
#     "european_aqi_rolling_min_168h",
#     "pm2_5_rolling_mean_3h",
#     "pm2_5_rolling_mean_6h",
#     "pm2_5_rolling_mean_12h",
#     "pm2_5_rolling_min_3h",
#     "pm2_5_rolling_min_6h",
#     "pm10_rolling_mean_12h",
#     "pm10_rolling_mean_24h",
#     "pm10_rolling_mean_48h",
#     "pm10_rolling_std_24h",
#     "pm10_rolling_std_168h",
#     "pm10_rolling_min_3h",
#     "nitrogen_dioxide_lag_6h",
#     "wind_speed_10m_rolling_std_168h",
#     "wind_speed_10m_rolling_min_48h",
#     "wind_speed_10m_rolling_mean_24h",
#     "wind_speed_10m_rolling_mean_168h",
#     "wind_speed_10m_rolling_min_168h",
#     "relative_humidity_2m_rolling_min_24h",
#     "relative_humidity_2m_rolling_std_24h",
#     "precipitation_cumulative_72h",
#     "days_since_last_rain",
#     "oxidant_index",
#     "pollutant_composite_index",
#     "epa_pm25_subindex",
#     "epa_pm10_subindex",
#     "pressure_change_6h",
#     "hour_sin",
#     "hour_cos",
#     "day_of_week_sin",
#     "day_of_week_cos",
#     "daylight_hours",
#     "is_evening_rush",
#     "hour_traffic_weight",
#     "weekend_traffic_factor",
#     "solar_radiation_category",
#     "is_day",
# ]

# # Day2: predict hours 24-48 — anchor on lag_24h as the "current" state
# TOP_FEATURES_DAY2 = [
#     "european_aqi_lag_24h",        # most recent known AQI at prediction time
#     "european_aqi_lag_1h",         # kept for trend context
#     "european_aqi_rolling_std_24h",
#     "european_aqi_rolling_std_48h",
#     "european_aqi_rolling_min_168h",
#     "pm2_5_rolling_mean_12h",
#     "pm2_5_rolling_mean_6h",
#     "pm2_5_rolling_min_6h",
#     "pm10_rolling_mean_24h",
#     "pm10_rolling_mean_48h",
#     "pm10_rolling_std_24h",
#     "pm10_rolling_std_168h",
#     "nitrogen_dioxide_lag_6h",
#     "wind_speed_10m_rolling_std_168h",
#     "wind_speed_10m_rolling_mean_168h",
#     "wind_speed_10m_rolling_min_168h",
#     "wind_speed_10m_rolling_min_48h",
#     "relative_humidity_2m_rolling_min_24h",
#     "relative_humidity_2m_rolling_std_24h",
#     "precipitation_cumulative_72h",
#     "days_since_last_rain",
#     "oxidant_index",
#     "pollutant_composite_index",
#     "epa_pm25_subindex",
#     "epa_pm10_subindex",
#     "pressure_change_6h",
#     "hour_sin",
#     "hour_cos",
#     "day_of_week_sin",
#     "day_of_week_cos",
#     "daylight_hours",
#     "weekend_traffic_factor",
#     "solar_radiation_category",
# ]

# # Day3: predict hours 48-72 — anchor on lag_48h as the "current" state
# TOP_FEATURES_DAY3 = [
#     "european_aqi_lag_48h",        # most recent known AQI at prediction time
#     "european_aqi_lag_24h",        # yesterday's level for trend
#     "european_aqi_lag_1h",         # kept for baseline context
#     "european_aqi_rolling_std_48h",
#     "european_aqi_rolling_min_168h",
#     "pm2_5_rolling_mean_12h",
#     "pm10_rolling_mean_48h",
#     "pm10_rolling_std_168h",
#     "wind_speed_10m_rolling_std_168h",
#     "wind_speed_10m_rolling_mean_168h",
#     "wind_speed_10m_rolling_min_168h",
#     "relative_humidity_2m_rolling_min_24h",
#     "precipitation_cumulative_72h",
#     "days_since_last_rain",
#     "oxidant_index",
#     "pollutant_composite_index",
#     "epa_pm25_subindex",
#     "epa_pm10_subindex",
#     "pressure_change_6h",
#     "hour_sin",
#     "hour_cos",
#     "day_of_week_sin",
#     "day_of_week_cos",
#     "daylight_hours",
#     "weekend_traffic_factor",
#     "solar_radiation_category",
# ]

ARTIFACTS_DIR = Path("/content/debug_exports")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

horizon = 3
DAILY_HOURS = 24
best_window_days = 180

SEED = 42
SPLIT_RATIO = 0.9
TARGET_MODEL_COUNT = 10
MAX_DL_MODELS = 2
DL_TYPES = {"gru", "lstm"}
TREE_MODEL_NAMES = {"lightgbm", "xgboost", "catboost", "random_forest", "extra_trees", "gradient_boosting"}

CPU_OVERRIDES = {
    "xgboost": {"tree_method": "hist", "n_jobs": 1, "random_state": SEED},
    "catboost": {"task_type": "CPU", "thread_count": 1, "random_seed": SEED},
    "extra_trees": {"n_jobs": 1, "random_state": SEED},
    "random_forest": {"n_jobs": 1, "random_state": SEED},
    "lightgbm": {"device_type": "cpu", "n_jobs": 1, "random_state": SEED},
    "gradient_boosting": {"random_state": SEED},
}

BENCHMARK_MODEL_NAMES = [config.name for config in MODEL_CONFIGS]

def apply_cpu_overrides(config: ModelConfig) -> ModelConfig:
    overrides = CPU_OVERRIDES.get(config.name, {})
    if not overrides:
        return config
    params = {**config.params, **overrides}
    return ModelConfig(name=config.name, type=config.type, params=params)


def get_mongo_client() -> MongoClient:
    uri = os.getenv("MONGO_URI")
    if not uri:
        raise ValueError("MONGO_URI is required")
    return MongoClient(uri)


def get_database():
    client = get_mongo_client()
    db_name = os.getenv("MONGO_DB_NAME", "aqi_predictor")
    return client.get_database(db_name)


def evaluate_forecast(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"rmse": rmse, "mae": mae, "r2": r2}


def per_horizon_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    metrics = {}
    labels = ["day1", "day2", "day3"]

    for idx, label in enumerate(labels):
        if idx < y_true.shape[1]:
            yt = y_true[:, idx]
            yp = y_pred[:, idx]
            metrics[f"rmse_{label}"] = float(np.sqrt(mean_squared_error(yt, yp)))
            metrics[f"mae_{label}"]  = float(mean_absolute_error(yt, yp))
            metrics[f"r2_{label}"]   = float(r2_score(yt, yp))

    return metrics

def _build_dl_model(model_type: str, input_shape: Tuple[int, int], output_steps: int, units: int):
    model = tf.keras.Sequential()
    if model_type == "gru":
        model.add(tf.keras.layers.GRU(units, input_shape=input_shape, return_sequences=False))
    else:
        model.add(tf.keras.layers.LSTM(units, input_shape=input_shape, return_sequences=False))
    model.add(tf.keras.layers.Dense(128, activation="relu"))
    model.add(tf.keras.layers.Dense(output_steps))
    model.compile(optimizer="adam", loss="mse")
    return model


def _prepare_sequence_data(
    features: pd.DataFrame,
    target: pd.Series,
    lookback: int,
):
    X, y, start_indices = [], [], []

    values = features.values
    target_values = target.values

    required_future = 72

    for idx in range(lookback, len(features) - required_future):

        day1 = target_values[idx : idx + 24].mean()
        day2 = target_values[idx + 24 : idx + 48].mean()
        day3 = target_values[idx + 48 : idx + 72].mean()

        X.append(values[idx - lookback : idx])
        y.append([day1, day2, day3])
        start_indices.append(idx)

    return (
        np.array(X),
        np.array(y),
        np.array(start_indices),
    )


def _fit_single_output(config: ModelConfig, x_train: np.ndarray, y_train: np.ndarray):
    if config.type == "lgbm":
        base = lgb.LGBMRegressor(**config.params)
    elif config.type == "xgb":
        base = xgb.XGBRegressor(**config.params)
    elif config.type == "cat":
        base = cb.CatBoostRegressor(**config.params, verbose=False)
    elif config.type == "rf":
        base = RandomForestRegressor(**config.params)
    elif config.type == "extra":
        base = ExtraTreesRegressor(**config.params)
    elif config.type == "ridge":
        base = Ridge(**config.params)
    elif config.type == "linreg":
        base = LinearRegression(**config.params)
    else:
        base = GradientBoostingRegressor(**config.params)
    base.fit(x_train, y_train)
    return base

# ── Single unified feature set (10 features) ────────────────────
TOP_FEATURES = [
    "european_aqi_lag_72h",
    "european_aqi_lag_24h",
    "pm2_5_rolling_mean_24h",
    "pm10_rolling_mean_24h",
    "pollutant_composite_index",
    "oxidant_index",
    "nitrogen_dioxide_lag_6h",
    "wind_speed_10m_rolling_mean_24h",
    "days_since_last_rain",
    "atmospheric_stability_index",
]


def train_model(
    config: ModelConfig,
    day1_matrices: tuple,
    day2_matrices: tuple,
    day3_matrices: tuple,
    feature_frame: pd.DataFrame = None,
    target: pd.Series = None,
    split_index: int = None,
):
    x_tr1, y_tr1, x_vl1, y_vl1, _ = day1_matrices
    x_tr2, y_tr2, x_vl2, y_vl2, _ = day2_matrices
    x_tr3, y_tr3, x_vl3, y_vl3, _ = day3_matrices

    # ── Deep Learning ────────────────────────────────────────────
    if config.type in DL_TYPES:
        if feature_frame is None or target is None or split_index is None:
            raise ValueError("feature_frame, target, and split_index required for DL")

        lookback = 24
        X_seq, y_seq, start_indices = _prepare_sequence_data(feature_frame, target, lookback)

        train_mask = start_indices + 72 <= split_index
        val_mask   = start_indices >= split_index

        if not train_mask.any() or not val_mask.any():
            raise ValueError("Not enough sequences for DL split")

        X_tr, X_vl     = X_seq[train_mask], X_seq[val_mask]
        y_tr_s, y_vl_s = y_seq[train_mask], y_seq[val_mask]

        model = _build_dl_model(
            config.type,
            input_shape=(X_tr.shape[1], X_tr.shape[2]),
            output_steps=3,
            units=config.params.get("units", 64),
        )
        model.fit(
            X_tr, y_tr_s,
            epochs=config.params.get("epochs", 10),
            batch_size=config.params.get("batch_size", 32),
            verbose=0,
        )
        train_preds = model.predict(X_tr)
        val_preds   = model.predict(X_vl)
        # stack true values to match (n, 3) shape expected by metrics
        y_tr_true = y_tr_s
        y_vl_true = y_vl_s
        return model, train_preds, val_preds, y_tr_true, y_vl_true

    # ── Classical ML — fully independent per day ─────────────────
    m1 = _fit_single_output(config, x_tr1.values, y_tr1.values)
    m2 = _fit_single_output(config, x_tr2.values, y_tr2.values)
    m3 = _fit_single_output(config, x_tr3.values, y_tr3.values)

    # Predictions per day — shape (n,)
    pred_tr = np.column_stack([
        m1.predict(x_tr1.values),
        m2.predict(x_tr2.values),
        m3.predict(x_tr3.values),
    ])
    pred_vl = np.column_stack([
        m1.predict(x_vl1.values),
        m2.predict(x_vl2.values),
        m3.predict(x_vl3.values),
    ])

    # True values — shape (n, 3)
    y_tr_true = np.column_stack([y_tr1.values, y_tr2.values, y_tr3.values])
    y_vl_true = np.column_stack([y_vl1.values, y_vl2.values, y_vl3.values])

    return (m1, m2, m3), pred_tr, pred_vl, y_tr_true, y_vl_true

def build_window_frame(frame: pd.DataFrame, days: int) -> pd.DataFrame:
    latest_timestamp = frame["timestamp"].max()
    window_start = latest_timestamp - pd.Timedelta(days=days)
    return frame.loc[frame["timestamp"] >= window_start].sort_values("timestamp").reset_index(drop=True)


def prepare_training_matrices(
    feature_frame: pd.DataFrame,
    target: pd.Series,
    split_index: int,
):
    required_future = 72

    if feature_frame.empty or len(feature_frame) <= required_future:
        return None, None, None

    max_start = len(target) - required_future
    if max_start <= 0 or split_index <= required_future or split_index >= max_start:
        return None, None, None

    start_indices = np.arange(max_start)
    train_mask = start_indices + required_future <= split_index
    val_mask   = start_indices >= split_index

    if not train_mask.any() or not val_mask.any():
        return None, None, None

    # Build per-day targets once — shape (max_start,) each
    y_day1 = np.array([target.iloc[i      : i + 24].mean() for i in range(max_start)])
    y_day2 = np.array([target.iloc[i + 24 : i + 48].mean() for i in range(max_start)])
    y_day3 = np.array([target.iloc[i + 48 : i + 72].mean() for i in range(max_start)])

    def _split(feat_cols, y_vec):
        cols = [c for c in feat_cols if c in feature_frame.columns]
        x = feature_frame[cols].iloc[:max_start].copy()
        y = pd.Series(y_vec, index=x.index)
        return (
            x.iloc[train_mask],
            y.iloc[train_mask],
            x.iloc[val_mask],
            y.iloc[val_mask],
            cols,
        )

    day1_matrices = _split(TOP_FEATURES_DAY1, y_day1)
    day2_matrices = _split(TOP_FEATURES_DAY2, y_day2)
    day3_matrices = _split(TOP_FEATURES_DAY3, y_day3)

    return day1_matrices, day2_matrices, day3_matrices

db = get_database()
collection = db["aqi_features_rawalpindi"]
data = pd.DataFrame(list(collection.find()))

if data.empty:
    raise ValueError("aqi_features_rawalpindi is empty")

if "_id" in data.columns:
    data = data.drop(columns=["_id"])

data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
data = data.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

feature_cols_all = [column for column in data.columns if column != "timestamp"]
feature_cols = [feature for feature in TOP_FEATURES if feature in data.columns] or feature_cols_all

window_frame = build_window_frame(data, best_window_days)
split_index = int(len(window_frame) * SPLIT_RATIO)

if split_index <= 72:
    raise ValueError("Not enough rows to create a leakage-safe split")

target = window_frame["european_aqi"].astype(float)
feature_frame = window_frame[feature_cols].apply(pd.to_numeric, errors="coerce").ffill().fillna(0.0)
feature_frame = feature_frame.loc[:, feature_frame.nunique(dropna=True) > 1].copy()

day1_matrices, day2_matrices, day3_matrices = prepare_training_matrices(
    feature_frame, target, split_index
)

if day1_matrices is None:
    raise ValueError("Unable to prepare benchmark matrices for the selected window and split")

used_features = len(day1_matrices[4]) + len(day2_matrices[4]) + len(day3_matrices[4])
rows_after_prep = len(day1_matrices[0]) + len(day1_matrices[2])  # train + val rows


In [ ]:
# ── Single unified feature set (10 features) ────────────────────
TOP_FEATURES = [
    "european_aqi_lag_72h",
    "european_aqi_lag_24h",
    "pm2_5_rolling_mean_24h",
    "pm10_rolling_mean_24h",
    "pollutant_composite_index",
    "oxidant_index",
    "nitrogen_dioxide_lag_6h",
    "wind_speed_10m_rolling_mean_24h",
    "days_since_last_rain",
    "atmospheric_stability_index",
]

# ── Simplified training matrix builder ──────────────────────────
def prepare_training_matrices(
    feature_frame: pd.DataFrame,
    target: pd.Series,
    split_index: int,
):
    """
    Returns a single (x_train, y_train, x_val, y_val, feature_cols) tuple.
    y is shape (n, 3) — [day1_avg, day2_avg, day3_avg].
    Each row i predicts the 3 daily averages starting at hour i.
    """
    required_future = 72

    if feature_frame.empty or len(feature_frame) <= required_future:
        return None

    max_start = len(target) - required_future
    if max_start <= 0 or split_index <= required_future or split_index >= max_start:
        return None

    start_indices = np.arange(max_start)
    train_mask = (start_indices + required_future) <= split_index
    val_mask   = start_indices >= split_index

    if not train_mask.any() or not val_mask.any():
        return None

    # Build (n, 3) target matrix once
    y_matrix = np.column_stack([
        [target.iloc[i      : i + 24].mean() for i in range(max_start)],
        [target.iloc[i + 24 : i + 48].mean() for i in range(max_start)],
        [target.iloc[i + 48 : i + 72].mean() for i in range(max_start)],
    ])

    feat_cols = [c for c in TOP_FEATURES if c in feature_frame.columns]
    X = feature_frame[feat_cols].iloc[:max_start].copy()
    Y = pd.DataFrame(y_matrix, index=X.index, columns=["day1", "day2", "day3"])

    return (
        X.iloc[train_mask],
        Y.iloc[train_mask],
        X.iloc[val_mask],
        Y.iloc[val_mask],
        feat_cols,
    )


# ── Single-model trainer ─────────────────────────────────────────
def train_model(
    config: ModelConfig,
    matrices: tuple,                 # (x_tr, y_tr, x_vl, y_vl, feat_cols)
    feature_frame: pd.DataFrame = None,
    target: pd.Series = None,
    split_index: int = None,
):
    """
    Trains ONE model that outputs [day1_avg, day2_avg, day3_avg].
    Returns: model, train_preds (n,3), val_preds (n,3), y_train_true (n,3), y_val_true (n,3)
    """
    x_tr, y_tr, x_vl, y_vl, _ = matrices

    # ── Deep Learning ────────────────────────────────────────────
    if config.type in DL_TYPES:
        if feature_frame is None or target is None or split_index is None:
            raise ValueError("feature_frame, target, and split_index required for DL")

        lookback = 24
        X_seq, y_seq, start_indices = _prepare_sequence_data(feature_frame, target, lookback)

        train_mask = start_indices + 72 <= split_index
        val_mask   = start_indices >= split_index

        if not train_mask.any() or not val_mask.any():
            raise ValueError("Not enough sequences for DL split")

        X_tr_s, X_vl_s = X_seq[train_mask], X_seq[val_mask]
        y_tr_s, y_vl_s = y_seq[train_mask], y_seq[val_mask]

        model = _build_dl_model(
            config.type,
            input_shape=(X_tr_s.shape[1], X_tr_s.shape[2]),
            output_steps=3,
            units=config.params.get("units", 64),
        )
        model.fit(
            X_tr_s, y_tr_s,
            epochs=config.params.get("epochs", 10),
            batch_size=config.params.get("batch_size", 32),
            verbose=0,
        )
        return model, model.predict(X_tr_s), model.predict(X_vl_s), y_tr_s, y_vl_s

    # ── Classical ML — native multi-output or wrapped ────────────
    x_tr_vals = x_tr.values
    x_vl_vals = x_vl.values
    y_tr_vals = y_tr.values   # shape (n, 3)

    if config.type == "lgbm":
        # LightGBM handles multi-output natively via MultiOutputRegressor
        base = MultiOutputRegressor(lgb.LGBMRegressor(**config.params))
    elif config.type == "xgb":
        base = MultiOutputRegressor(xgb.XGBRegressor(**config.params))
    elif config.type == "cat":
        base = MultiOutputRegressor(cb.CatBoostRegressor(**config.params, verbose=False))
    elif config.type == "rf":
        base = RandomForestRegressor(**config.params)          # natively multi-output
    elif config.type == "extra":
        base = ExtraTreesRegressor(**config.params)            # natively multi-output
    elif config.type == "gbr":
        base = MultiOutputRegressor(GradientBoostingRegressor(**config.params))
    elif config.type == "ridge":
        base = Ridge(**config.params)                          # natively multi-output
    else:
        base = LinearRegression(**config.params)               # natively multi-output

    base.fit(x_tr_vals, y_tr_vals)

    train_preds = base.predict(x_tr_vals)   # (n_train, 3)
    val_preds   = base.predict(x_vl_vals)   # (n_val, 3)

    return base, train_preds, val_preds, y_tr_vals, y_vl.values


# ── Data loading (unchanged) ─────────────────────────────────────
db = get_database()
collection = db["aqi_features_rawalpindi"]
data = pd.DataFrame(list(collection.find()))

if data.empty:
    raise ValueError("aqi_features_rawalpindi is empty")

if "_id" in data.columns:
    data = data.drop(columns=["_id"])

data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
data = data.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

window_frame  = build_window_frame(data, best_window_days)
split_index   = int(len(window_frame) * SPLIT_RATIO)

if split_index <= 72:
    raise ValueError("Not enough rows to create a leakage-safe split")

target        = window_frame["european_aqi"].astype(float)
feature_frame = (
    window_frame[TOP_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .ffill()
    .fillna(0.0)
)
feature_frame = feature_frame.loc[:, feature_frame.nunique(dropna=True) > 1].copy()

matrices = prepare_training_matrices(feature_frame, target, split_index)

if matrices is None:
    raise ValueError("Unable to prepare training matrices for the selected window and split")

used_features    = len(matrices[4])
rows_after_prep  = len(matrices[0]) + len(matrices[2])   # train + val rows


# ── Benchmark loop ───────────────────────────────────────────────
results = []
model_lookup = {config.name: config for config in MODEL_CONFIGS}

benchmark_models = [
    apply_cpu_overrides(model_lookup[name])
    for name in BENCHMARK_MODEL_NAMES
    if name in model_lookup
]

if not benchmark_models:
    raise ValueError("No benchmark models available")

for config in benchmark_models:
    print(f"\nRunning benchmark for {config.name}...")
    try:
        model, train_preds, val_preds, y_train_true, y_val_true = train_model(
            config,
            matrices,
            feature_frame=feature_frame,
            target=target,
            split_index=split_index,
        )

        train_metrics = evaluate_forecast(y_train_true, train_preds)
        train_metrics.update(per_horizon_metrics(y_train_true, train_preds))

        val_metrics = evaluate_forecast(y_val_true, val_preds)
        val_metrics.update(per_horizon_metrics(y_val_true, val_preds))

        print(
            f"{config.name} | "
            f"Train RMSE: {train_metrics['rmse']:.4f} | "
            f"Val RMSE:   {val_metrics['rmse']:.4f} | "
            f"Val R²:     {val_metrics['r2']:.4f} | "
            f"Day1: {val_metrics.get('r2_day1', float('nan')):.4f} | "
            f"Day2: {val_metrics.get('r2_day2', float('nan')):.4f} | "
            f"Day3: {val_metrics.get('r2_day3', float('nan')):.4f}"
        )

        results.append({
            "model":           config.name,
            "type":            config.type,
            "window_days":     best_window_days,
            "rows_after_prep": rows_after_prep,
            "features_used":   used_features,
            "train_rmse":      train_metrics["rmse"],
            "train_mae":       train_metrics["mae"],
            "train_r2":        train_metrics["r2"],
            "train_r2_day1":   train_metrics.get("r2_day1"),
            "train_r2_day2":   train_metrics.get("r2_day2"),
            "train_r2_day3":   train_metrics.get("r2_day3"),
            "val_rmse":        val_metrics["rmse"],
            "val_mae":         val_metrics["mae"],
            "val_r2":          val_metrics["r2"],
            "val_r2_day1":     val_metrics.get("r2_day1"),
            "val_r2_day2":     val_metrics.get("r2_day2"),
            "val_r2_day3":     val_metrics.get("r2_day3"),
            "overfit_gap_rmse": val_metrics["rmse"] - train_metrics["rmse"],
        })

    except Exception as exc:
        print(f"Model {config.name} failed: {exc}")
        results.append({
            "model": config.name, "type": config.type,
            "window_days": best_window_days, "rows_after_prep": rows_after_prep,
            "features_used": used_features,
            **{k: float("nan") for k in [
                "train_rmse","train_mae","train_r2","train_r2_day1","train_r2_day2","train_r2_day3",
                "val_rmse","val_mae","val_r2","val_r2_day1","val_r2_day2","val_r2_day3","overfit_gap_rmse",
            ]},
            "error": str(exc),
        })

results_df   = pd.DataFrame(results)
results_path = ARTIFACTS_DIR / "model_benchmark_metrics.csv"
results_df.to_csv(results_path, index=False)

valid_results = results_df.dropna(subset=["val_rmse"]).sort_values(
    ["val_rmse", "model"], na_position="last"
).reset_index(drop=True)

if valid_results.empty:
    raise ValueError("All benchmark runs failed")

best_row            = valid_results.iloc[0]
best_model_name     = str(best_row["model"])

best_tree_results   = valid_results[valid_results["model"].isin(TREE_MODEL_NAMES)]
best_tree_model_name = str(best_tree_results.iloc[0]["model"]) if not best_tree_results.empty else best_model_name

(ARTIFACTS_DIR / "best_model_name.txt").write_text(best_model_name,      encoding="utf-8")
(ARTIFACTS_DIR / "best_tree_model_name.txt").write_text(best_tree_model_name, encoding="utf-8")

with open(ARTIFACTS_DIR / "best_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_model_name":  best_model_name,
        "best_model_type":  str(best_row["type"]),
        "best_window_days": best_window_days,
        "val_rmse":         float(best_row["val_rmse"]),
        "val_mae":          float(best_row["val_mae"]),
        "val_r2":           float(best_row["val_r2"]),
        "val_r2_day1":      float(best_row.get("val_r2_day1", float("nan"))),
        "val_r2_day2":      float(best_row.get("val_r2_day2", float("nan"))),
        "val_r2_day3":      float(best_row.get("val_r2_day3", float("nan"))),
        "rows_after_prep":  int(best_row["rows_after_prep"]),
        "features_used":    int(best_row["features_used"]),
    }, f, indent=2)

display(results_df)
print(f"\nSaved {results_path}")
print(f"Best model (by val RMSE): {best_model_name}")
print(f"Best tree model:          {best_tree_model_name}")


Running benchmark for lightgbm...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

lightgbm | Train RMSE: 4.1715 | Val RMSE:   15.0580 | Val R²:     0.1550 | Day1: 0.6351 | Day2: 0.0720 | Day3: -0.2420

Running benchmark for xgboost...
xgboost | Train RMSE: 1.2306 | Val RMSE:   15.5998 | Val R²:     0.1068 | Day1: 0.7511 | Day2: -0.1240 | Day3: -0.3065

Running benchmark for catboost...
catboost | Train RMSE: 2.6888 | Val RMSE:   15.1209 | Val R²:     0.1591 | Day1: 0.7206 | Day2: 0.0759 | Day3: -0.3194

Running benchmark for random_forest...
random_forest | Train RMSE: 0.7642 | Val RMSE:   16.3008 | Val R²:     0.0110 | Day1: 0.5760 | Day2: -0.0612 | Day3: -0.4818

Running benchmark for extra_trees...
extra_trees | Train RMSE: 0.0545 | Val RMSE:   14.6035 | Val R²:     0.2095 | Day1: 0.6896 | Day2: 0.1352 | Day3: -0.1963

Running benchmark for gradient_boosting...
gradient_boosting | Train RMSE: 4.2713 | Val RMSE:   14.8407 | Val R²:     0.1931 | Day1: 0.7559 | Day2: 0.1156 | Day3: -0.2921

Running benchmark for ridge_regression...
ridge_regression | Train RMSE: 7.9

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
gru | Train RMSE: 7.4484 | Val RMSE:   14.4941 | Val R²:     0.2085 | Day1: 0.5871 | Day2: 0.1544 | Day3: -0.1161

Running benchmark for lstm...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
lstm | Train RMSE: 6.9239 | Val RMSE:   14.7707 | Val R²:     0.1750 | Day1: 0.5632 | Day2: 0.0522 | Day3: -0.0903


,model,type,window_days,rows_after_prep,features_used,train_rmse,train_mae,train_r2,train_r2_day1,train_r2_day2,train_r2_day3,val_rmse,val_mae,val_r2,val_r2_day1,val_r2_day2,val_r2_day3,overfit_gap_rmse
0,lightgbm,lgbm,180,4178,10,4.171480,3.030026,0.924802,0.968993,0.909833,0.895579,15.058028,10.463952,0.155042,0.635116,0.071961,-0.241952,10.886548
1,xgboost,xgb,180,4178,10,1.230621,0.817053,0.993459,0.996723,0.992526,0.991128,15.599753,11.003383,0.106841,0.751060,-0.124047,-0.306490,14.369133
2,catboost,cat,180,4178,10,2.688811,1.926626,0.968770,0.985755,0.957980,0.962575,15.120945,10.586956,0.159052,0.720612,0.075921,-0.319378,12.432134
3,random_forest,rf,180,4178,10,0.764236,0.387801,0.997478,0.998656,0.996991,0.996788,16.300789,11.859480,0.011004,0.576042,-0.061219,-0.481809,15.536553
4,extra_trees,extra,180,4178,10,0.054457,0.023164,0.999987,0.999987,0.999986,0.999989,14.603522,10.403823,0.209485,0.689559,0.135194,-0.196298,14.549065
5,gradient_boosting,gbr,180,4178,10,4.271296,3.062393,0.921157,0.968272,0.903599,0.891599,14.840722,10.239789,0.193132,0.755867,0.115648,-0.292120,10.569426
6,ridge_regression,ridge,180,4178,10,7.985078,5.824528,0.724273,0.913783,0.679163,0.579873,14.825685,10.235783,0.193746,0.748625,0.114588,-0.281977,6.840607
7,linear_regression,linreg,180,4178,10,7.985078,5.824513,0.724273,0.913783,0.679163,0.579873,14.826140,10.236036,0.193697,0.748644,0.114430,-0.281982,6.841062
8,gru,gru,180,4178,10,7.448421,5.638039,0.760623,0.876298,0.742236,0.663333,14.494136,10.507316,0.208454,0.587061,0.154423,-0.116123,7.045715
9,lstm,lstm,180,4178,10,6.923905,5.118888,0.793142,0.898819,0.777676,0.702933,14.770663,10.299879,0.175021,0.563222,0.052170,-0.090329,7.846757



Saved /content/debug_exports/model_benchmark_metrics.csv
Best model (by val RMSE): gru
Best tree model:          extra_trees
